# DSPy 101 — Replacing Prompts with Signatures

**Week 6 | Notebook 1 of 12**

**What you'll learn:**
- LM configuration (OpenAI + Ollama)
- Your first Signature — QuestionAnswering
- dspy.Predict — basic prediction
- dspy.ChainOfThought — adding rationale
- dspy.ProgramOfThought — math problems with code execution
- Side-by-side output comparison across module types
- Inline vs class-based signature syntax

**Runtime:** ~20 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/01_signatures_modules.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/01_signatures_modules.ipynb
Task:      Signatures and core modules
Calls:     ~15

With GPT-4o:       $0.15 USD
With GPT-4o-mini:  $0.01 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup & LM Configuration

In [2]:
import dspy

from src.config import get_dspy_lm, print_config

print_config()

# Configure DSPy with our unified LM
lm = get_dspy_lm()
dspy.configure(lm=lm)

print(f"\n✅ DSPy configured with: {lm.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      groq
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.6-flash
  groq model:      openai/gpt-oss-120b
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: groq/openai/gpt-oss-120b


## 2. Your First Signature — QuestionAnswering

In [3]:
# Class-based signature (recommended for production)
class QuestionAnswering(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="A concise, factual answer")


# Inline signature (quick prototyping)
# inline_sig = "question -> answer"

print("✅ Signature defined")
print(f"  Input: {QuestionAnswering.fields.keys()}")

✅ Signature defined
  Input: dict_keys(['question', 'answer'])


## 3. dspy.Predict — Basic Prediction

In [4]:
# Basic prediction module
predictor = dspy.Predict(QuestionAnswering)

result = predictor(question="What is the capital of India?")
print("Question: What is the capital of India?")
print(f"Answer: {result.answer}")

Question: What is the capital of India?
Answer: New Delhi


## 4. dspy.ChainOfThought — Adding Rationale

In [6]:
# ChainOfThought automatically adds a 'reasoning' field (dspy 3.x renamed 'rationale')
cot = dspy.ChainOfThought(QuestionAnswering)

result = cot(question="If a train travels 60 km/h for 2.5 hours, how far does it go?")

print("Rationale:")
print(result.reasoning)
print(f"\nAnswer: {result.answer}")

Rationale:
Distance is calculated by multiplying speed by time.  
\(60\text{ km/h} \times 2.5\text{ h} = 150\text{ km}\).

Answer: 150 km


## 5. dspy.ProgramOfThought — Math with Code Execution

In [7]:
# ProgramOfThought generates and executes Python code
pot = dspy.ProgramOfThought(QuestionAnswering)

result = pot(question="What is the average of 45, 67, 89, and 23?")

print(f"Answer: {result.answer}")
print("\nThis answer was computed by generated Python code, not guessed by the LLM.")

Answer: 56

This answer was computed by generated Python code, not guessed by the LLM.


In [8]:
import dspy
from src.config import get_dspy_lm
# Configure DSPy using your project's .env settings
class QuestionAnswering(dspy.Signature):
    """Answer a question using generated Python code."""
    question = dspy.InputField()
    answer = dspy.OutputField()
# Generate and execute Python code
pot = dspy.ProgramOfThought(QuestionAnswering)
result = pot(
    question="What is the average of 45, 67, 89, and 23?"
)
print("Final result:")
print(result)
print(f"\nAnswer: {result.answer}")
print("\nGenerated code and execution history:")
dspy.inspect_history(n=5)


Final result:
Prediction(
    reasoning='The code calculates the average by summing the numbers\u202f45,\u202f67,\u202f89,\u202f23, which gives 224, then dividing by the count of numbers (4). 224\u202f÷\u202f4\u202f=\u202f56. Hence the average is 56.',
    answer='56'
)

Answer: 56

Generated code and execution history:




[2026-10-07T07:54:03.136358]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): A concise, factual answer
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer questions with short factual responses.


User message:

[[ ## question ## ]]
If a train travels 60 km/h for 2.5 hours, how far does it go?

Respond with the corresponding output fields, starting with the fie

## 6. Side-by-Side Output Comparison

In [9]:
question = "Explain transformer attention in one sentence."

modules = {
    "Predict": dspy.Predict(QuestionAnswering),
    "ChainOfThought": dspy.ChainOfThought(QuestionAnswering),
    "ProgramOfThought": dspy.ProgramOfThought(QuestionAnswering),
}

print(f"Question: {question}\n")
for name, module in modules.items():
    result = module(question=question)
    print(f"--- {name} ---")
    if hasattr(result, "reasoning"):
        print(f"Reasoning: {result.reasoning[:100]}...")
    print(f"Answer: {result.answer}\n")

Question: Explain transformer attention in one sentence.

--- Predict ---
Answer: Transformer attention computes weighted sums of token representations by measuring pairwise similarity between query and key vectors, allowing each token to selectively focus on relevant parts of the input sequence.

--- ChainOfThought ---
Reasoning: The user asks for a one‑sentence explanation of transformer attention, so I must craft a brief, clea...
Answer: Transformer attention computes each token’s representation as a weighted sum of all token values, where the weights are determined by the similarity between that token’s query vector and the keys of all tokens.



2026/10/07 08:16:01 ERROR dspy.predict.program_of_thought: Error in code execution: TypeError: ["SUBMIT() got an unexpected keyword argument 'answer'"]
2026/10/07 08:16:03 WARNING dspy.predict.predict: Input contains fields not in signature. These fields will be ignored: ['previous_code', 'error']. Expected fields: ['question', 'final_generated_code', 'code_output'].


--- ProgramOfThought ---
Reasoning: The generated code defines the answer as a concise one‑sentence description of transformer attention...
Answer: Transformer attention computes a weighted sum of value vectors where the weights are derived from the similarity between query and key vectors.



## 7. Inline vs Class-Based Signature Syntax

In [10]:
# Class-based (recommended)
class SentimentClassification(dspy.Signature):
    """Classify the sentiment of the given text."""

    text: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")
    confidence: float = dspy.OutputField(desc="confidence score between 0 and 1")


classifier = dspy.Predict(SentimentClassification)
result = classifier(text="This product is amazing!")
print(f"Class-based: {result.sentiment} (confidence: {result.confidence})")


#------------------------------------------------
# Inline (quick prototyping)
inline_classifier = dspy.Predict("text -> sentiment, confidence")
result2 = inline_classifier(text="This product is amazing!")
print(f"Inline: {result2.sentiment} (confidence: {result2.confidence})")

Class-based: positive (confidence: 0.99)
Inline: positive (confidence: 0.99)


In [11]:
from typing import Literal

Sentiment = Literal["positive", "negative", "neutral"]

Sig = dspy.Signature(
    "text -> sentiment:Sentiment, confidence: float",
    instructions=(
        "Classify the sentiment of the given text. "
        "Sentiment must be 'positive', 'negative', or 'neutral'. "
        "Confidence is a score between 0 and 1."
    ),
)
inline_classifier = dspy.Predict(Sig)
result = inline_classifier(text="This product is amazing!")
print(f"Inline: {result.sentiment} (confidence: {result.confidence})")

Inline: positive (confidence: 0.99)
